# Model Context Protocol (MCP)

What MCP is not
- A framework for building agents
- A fundamental change to how agent work
- A way to code agents

What MCP is
- A protocol - a standard
- A simple way to integrate tools, resources, and prompts
- "A UCB-C port for AI application"

## MCP Core Concepts
The Three components of MCP:
- **Host** is an LLM app like Claude or our Agent architecture
- **MCP Client** lives inside Host and connects 1:1 to MCP Server
- **MCP Server** provides tools, context, and prompts

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

NODE_BIN = "/usr/local/opt/node@20/bin/node"
NPX_CLI = "/usr/local/opt/node@20/lib/node_modules/npm/bin/npx-cli.js"

In [2]:
load_dotenv(override=True)

True

### Using MCP in OpenAI Agent SDK
1. Create a Client
2. Have it spawn a server
3. Collect the tools that the server can use

In [3]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}


async with MCPServerStdio(
    params=fetch_params, client_session_timeout_seconds=60
) as server:
    fetch_tools = await server.list_tools()

fetch_tools

[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get 

In [4]:
playwright_params = {
    "command": NODE_BIN,
    "args": [NPX_CLI, "--yes", "@playwright/mcp@latest"],
    "env": {"PATH": f"/usr/local/opt/node@20/bin:{os.environ.get('PATH', '')}"},
}

async with MCPServerStdio(
    params=playwright_params, client_session_timeout_seconds=120
) as server:
    playwright_tools = await server.list_tools()

playwright_tools

[Tool(name='browser_close', title=None, description='Close the page', inputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'object', 'properties': {}, 'additionalProperties': False}, outputSchema=None, icons=None, annotations=ToolAnnotations(title='Close browser', readOnlyHint=False, destructiveHint=True, idempotentHint=None, openWorldHint=True), meta=None, execution=None),
 Tool(name='browser_resize', title=None, description='Resize the browser window', inputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'object', 'properties': {'width': {'type': 'number', 'description': 'Width of the browser window'}, 'height': {'type': 'number', 'description': 'Height of the browser window'}}, 'required': ['width', 'height'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=ToolAnnotations(title='Resize browser window', readOnlyHint=False, destructiveHint=True, idempotentHint=None, openWorldHint=True), meta=None, execut

In [5]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {
    "command": NODE_BIN,
    "args": [NPX_CLI, "-y", "@modelcontextprotocol/server-filesystem", sandbox_path],
    "env": {"PATH": f"/usr/local/opt/node@20/bin:{os.environ.get('PATH', '')}"},
}

async with MCPServerStdio(
    params=files_params, client_session_timeout_seconds=60
) as server:
    file_tools = await server.list_tools()

file_tools

Error initializing MCP server: Connection closed


McpError: Connection closed

In [ ]:
instructions = """
You browse the internet to accomplish your instructions.
You are highly capable at browsing the internet independently to accomplish your task, 
including accepting all cookies and clicking 'not now' as
appropriate to get to the content you need. If one website isn't fruitful, try another. 
Be persistent until you have solved your assignment,
trying different options and sites as needed.
When you need to write files, you do that inside the sandbox folder only.
"""


async with MCPServerStdio(
    params=files_params, client_session_timeout_seconds=60
) as mcp_server_files:
    async with MCPServerStdio(
        params=playwright_params, client_session_timeout_seconds=60
    ) as mcp_server_browser:
        agent = Agent(
            name="investigator",
            instructions=instructions,
            model="gpt-4.1-mini",
            mcp_servers=[mcp_server_files, mcp_server_browser],
        )
        with trace("investigate"):
            result = await Runner.run(
                agent,
                "Find a great recipe for Banoffee Pie, then summarize it in markdown to banoffee.md",
            )
            print(result.final_output)